# RQ6 – Robustness and Generalisation

**Research Question:** How robust is the best supervised learning model under different train-test splits, cross-validation settings, and data perturbation scenarios (noise, missingness)?

**Dataset:** [Kaggle Link](https://www.kaggle.com/datasets/imranalishahh/marketing-and-product-performance-dataset)

In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
import os

os.makedirs('figures', exist_ok=True)
os.makedirs('tables', exist_ok=True)

import glob, os

# Auto-detect the dataset file anywhere in the Colab/Kaggle environment
_search_paths = [
    '/kaggle/input/marketing-and-product-performance-dataset/',
    '/content/sample_data/',
    '/content/',
    '/content/drive/MyDrive/',
    '.',
]
_extensions = ['*.csv', '*.xlsx', '*.xls']
DATA_PATH = None

for _dir in _search_paths:
    for _ext in _extensions:
        _matches = glob.glob(os.path.join(_dir, '**', _ext), recursive=True) + glob.glob(os.path.join(_dir, _ext))
        _matches = [m for m in _matches if 'marketing' in m.lower() or 'product' in m.lower() or 'performance' in m.lower()]
        if _matches:
            DATA_PATH = _matches[0]
            break
    if DATA_PATH:
        break

# Final fallback: pick any CSV/Excel in /content
if DATA_PATH is None:
    for _ext in _extensions:
        _all = glob.glob(f'/content/**/{_ext}', recursive=True) + glob.glob(f'./**/{_ext}', recursive=True)
        if _all:
            DATA_PATH = _all[0]
            break

if DATA_PATH:
    print(f'Dataset found: {DATA_PATH}')
else:
    raise FileNotFoundError(
        'Dataset not found. Please upload the CSV/Excel file to Colab via:\n'
        '  Files panel (left sidebar) → Upload, then re-run this cell.')
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

Dataset found: /content/marketing_and_product_performance 2.csv


In [2]:
# ── Load & Preprocess ─────────────────────────────────────────────────────────
try:
    df = pd.read_csv(DATA_PATH)
except Exception:
    df = pd.read_excel(DATA_PATH)

candidates = [c for c in df.columns if any(k in c.lower() for k in ['revenue','sales','sale','income','profit','amount'])]
TARGET = candidates[0] if candidates else df.select_dtypes(include=np.number).var().idxmax()

df_clean = df.dropna(thresh=len(df)*0.5, axis=1).copy()
X_raw = df_clean.drop(columns=[TARGET]).copy()
y = df_clean[TARGET].reset_index(drop=True)

for col in X_raw.select_dtypes(include='object').columns:
    X_raw[col] = LabelEncoder().fit_transform(X_raw[col].astype(str))

X_clean = SimpleImputer(strategy='median').fit_transform(X_raw)
X_scaled = StandardScaler().fit_transform(X_clean)

MODEL = RandomForestRegressor(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1)

def get_metrics(X_tr, X_te, y_tr, y_te):
    MODEL.fit(X_tr, y_tr)
    p = MODEL.predict(X_te)
    return (round(mean_absolute_error(y_te,p),4),
            round(np.sqrt(mean_squared_error(y_te,p)),4),
            round(r2_score(y_te,p),4))

print('Preprocessing done.')

Preprocessing done.


In [3]:
# ── Scenario 1: Standard 80/20 split ─────────────────────────────────────────
Xtr, Xte, ytr, yte = train_test_split(X_scaled, y, test_size=0.2, random_state=RANDOM_STATE)
s1 = get_metrics(Xtr, Xte, ytr, yte)
print('Standard split:', s1)

Standard split: (25019.0251, np.float64(28964.8929), -0.0289)


In [4]:
# ── Scenario 2: 5-Fold Cross-Validation ──────────────────────────────────────
kf5 = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv5_r2   = cross_val_score(MODEL, X_scaled, y, cv=kf5, scoring='r2', n_jobs=-1)
cv5_mae  = -cross_val_score(MODEL, X_scaled, y, cv=kf5, scoring='neg_mean_absolute_error', n_jobs=-1)
cv5_rmse = np.sqrt(-cross_val_score(MODEL, X_scaled, y, cv=kf5, scoring='neg_mean_squared_error', n_jobs=-1))
s2 = (round(cv5_mae.mean(),4), round(cv5_rmse.mean(),4), round(cv5_r2.mean(),4))
s2_std = round(cv5_r2.std(), 4)
print('5-fold CV:', s2, '±', s2_std)

5-fold CV: (np.float64(25041.0037), np.float64(28933.449), np.float64(-0.0281)) ± 0.0051


In [5]:
# ── Scenario 3: 10-Fold Cross-Validation ─────────────────────────────────────
kf10 = KFold(n_splits=10, shuffle=True, random_state=RANDOM_STATE)
cv10_r2   = cross_val_score(MODEL, X_scaled, y, cv=kf10, scoring='r2', n_jobs=-1)
cv10_mae  = -cross_val_score(MODEL, X_scaled, y, cv=kf10, scoring='neg_mean_absolute_error', n_jobs=-1)
cv10_rmse = np.sqrt(-cross_val_score(MODEL, X_scaled, y, cv=kf10, scoring='neg_mean_squared_error', n_jobs=-1))
s3 = (round(cv10_mae.mean(),4), round(cv10_rmse.mean(),4), round(cv10_r2.mean(),4))
s3_std = round(cv10_r2.std(), 4)
print('10-fold CV:', s3, '±', s3_std)

10-fold CV: (np.float64(24978.537), np.float64(28849.5417), np.float64(-0.0228)) ± 0.0068


In [6]:
# ── Scenario 4: 10% Gaussian Noise Injection ──────────────────────────────────
X_noisy10 = X_scaled + np.random.normal(0, 0.1 * X_scaled.std(axis=0), X_scaled.shape)
Xtr4, Xte4, ytr4, yte4 = train_test_split(X_noisy10, y, test_size=0.2, random_state=RANDOM_STATE)
s4 = get_metrics(Xtr4, Xte4, ytr4, yte4)
print('10% noise:', s4)

10% noise: (24779.3229, np.float64(28839.6661), -0.02)


In [7]:
# ── Scenario 5: 20% Simulated Missingness ────────────────────────────────────
X_missing = X_scaled.copy()
mask = np.random.rand(*X_missing.shape) < 0.20
X_missing[mask] = np.nan
X_missing = SimpleImputer(strategy='mean').fit_transform(X_missing)
Xtr5, Xte5, ytr5, yte5 = train_test_split(X_missing, y, test_size=0.2, random_state=RANDOM_STATE)
s5 = get_metrics(Xtr5, Xte5, ytr5, yte5)
print('20% missing:', s5)

20% missing: (24986.4978, np.float64(28933.5298), -0.0266)


In [8]:
# ── Compile Results ───────────────────────────────────────────────────────────
results_df = pd.DataFrame([
    {'Scenario': 'Standard 80/20 Split', 'MAE': s1[0], 'RMSE': s1[1], 'R2': s1[2], 'Std Dev R2': 0.0},
    {'Scenario': '5-Fold CV',            'MAE': s2[0], 'RMSE': s2[1], 'R2': s2[2], 'Std Dev R2': s2_std},
    {'Scenario': '10-Fold CV',           'MAE': s3[0], 'RMSE': s3[1], 'R2': s3[2], 'Std Dev R2': s3_std},
    {'Scenario': '10% Noise Added',      'MAE': s4[0], 'RMSE': s4[1], 'R2': s4[2], 'Std Dev R2': 0.0},
    {'Scenario': '20% Missingness',      'MAE': s5[0], 'RMSE': s5[1], 'R2': s5[2], 'Std Dev R2': 0.0},
])
results_df.to_csv('tables/RQ6_robustness.csv', index=False)
print('Table saved.')
results_df

Table saved.


,Scenario,MAE,RMSE,R2,Std Dev R2
0,Standard 80/20 Split,25019.0251,28964.8929,-0.0289,0.0000
1,5-Fold CV,25041.0037,28933.4490,-0.0281,0.0051
2,10-Fold CV,24978.5370,28849.5417,-0.0228,0.0068
3,10% Noise Added,24779.3229,28839.6661,-0.0200,0.0000
4,20% Missingness,24986.4978,28933.5298,-0.0266,0.0000


In [9]:
# ── Figure: Line Chart of R² Across Scenarios ────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
colors = ['#2E4057','#048A81','#54C6EB']

for ax, metric, color in zip(axes, ['MAE','RMSE','R2'], colors):
    vals = results_df[metric].values
    x = np.arange(len(results_df))
    ax.plot(x, vals, marker='o', linewidth=2.5, markersize=10, color=color)
    ax.fill_between(x, vals*0.95, vals*1.05, alpha=0.15, color=color)
    ax.set_xticks(x)
    ax.set_xticklabels(results_df['Scenario'], rotation=20, ha='right', fontsize=9)
    ax.set_title(f'{metric} Across Scenarios', fontweight='bold')
    ax.set_ylabel(metric)
    ax.grid(alpha=0.3)
    ax.spines[['top','right']].set_visible(False)

fig.suptitle('Figure 6. Model Robustness Under Different Experimental Conditions\n(Marketing and Product Performance Dataset)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('figures/RQ6_robustness.pdf', dpi=300, bbox_inches='tight')
plt.show()
print('Figure saved.')

Figure saved.


In [10]:
# ── Boxplot of CV Scores ──────────────────────────────────────────────────────
fig2, ax2 = plt.subplots(figsize=(8, 5))
cv_data = [cv5_r2, cv10_r2]
bp = ax2.boxplot(cv_data, patch_artist=True, widths=0.4,
                 boxprops=dict(facecolor='#048A81', alpha=0.7),
                 medianprops=dict(color='#2E4057', linewidth=2))
ax2.set_xticklabels(['5-Fold CV', '10-Fold CV'], fontsize=12)
ax2.set_ylabel('R² Score', fontsize=12)
ax2.set_title('Figure 6b. Cross-Validation Score Distribution', fontsize=13, fontweight='bold')
ax2.grid(axis='y', alpha=0.3)
ax2.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig('figures/RQ6_cv_boxplot.pdf', dpi=300, bbox_inches='tight')
plt.show()
print('Boxplot saved.')

Boxplot saved.


## Summary

**RQ6** tests the best model (Random Forest) across 5 scenarios: standard split, 5-fold CV, 10-fold CV, 10% noise, and 20% missingness. Results in `tables/RQ6_robustness.csv` and `figures/RQ6_robustness.pdf`.